# 06 — Operator Hint Experiment (±5 VM/step)

Tests whether advance operator surge-hints reduce SLA breaches. A hint-aware
agent is trained on an environment with frequent synthetic surges, where the
3 hint slots activate a few steps **before** each surge. We then compare the
same agent evaluated **with hints active** vs **with hints blanked** (forced to
zero) on identical surge placements.

This is the default ±5 VM/step regime (the agent can scale quickly).

Uses `env.py`, `agent.py`.

## 1. Imports

In [1]:
import json
import numpy as np
import torch

from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic, train_ppo

stats = json.load(open('trace_params.json'))['stats']
print("Imports ready.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Imports ready.


## 2. Train a hint-aware agent (dense surges)

Trained with 8–12 surges per episode so the hint signal appears often enough
for the agent to learn the association. **~15-20 min run.**

In [2]:
env_hint = CloudClusterEnv(stats, enable_surges=True, enable_hints=True,
                           min_surges=8, max_surges=12, seed=None)
net_hint = ActorCritic()
optimizer = torch.optim.Adam(net_hint.parameters(), lr=3e-4)

print("Training hint-aware agent (±5 VM/step, dense surges + hints)...\n")
episode_rewards, convergence_log = train_ppo(env_hint, net_hint, optimizer,
                                             total_steps=250_000)

torch.save(net_hint.state_dict(), 'ppo_hint_aware.pth')
json.dump(convergence_log, open('ppo_hint_aware_convergence.json', 'w'))
print(f"\nSaved. First: {episode_rewards[0]:.1f}  Last: {episode_rewards[-1]:.1f}")

Training hint-aware agent (±5 VM/step, dense surges + hints)...

steps   2048 | recent ep reward   -162.8
steps   4096 | recent ep reward   -176.1
steps   6144 | recent ep reward   -181.3
steps   8192 | recent ep reward   -178.7
steps  10240 | recent ep reward   -170.7
steps  12288 | recent ep reward   -167.7
steps  14336 | recent ep reward   -176.3
steps  16384 | recent ep reward   -180.0
steps  18432 | recent ep reward   -179.1
steps  20480 | recent ep reward   -180.7
steps  22528 | recent ep reward   -186.5
steps  24576 | recent ep reward   -179.8
steps  26624 | recent ep reward   -187.8
steps  28672 | recent ep reward   -185.4
steps  30720 | recent ep reward   -188.8
steps  32768 | recent ep reward   -179.1
steps  34816 | recent ep reward   -175.3
steps  36864 | recent ep reward   -182.1
steps  38912 | recent ep reward   -184.7
steps  40960 | recent ep reward   -189.8
steps  43008 | recent ep reward   -203.4
steps  45056 | recent ep reward   -211.5
steps  47104 | recent ep reward  

## 3. Evaluate: with hints vs hints-blanked (same surges)

The same trained agent is run two ways on identical surge placements (same
seeds): once with the hint slots active, once with them forced to zero. The
difference in breaches isolates the hint's value.

In [3]:
def eval_hint(net, n_episodes, force_zero_hints, seed_base=3000):
    breaches_list = []
    for i in range(n_episodes):
        e = CloudClusterEnv(stats, enable_surges=True, enable_hints=True,
                            min_surges=8, max_surges=12, seed=seed_base+i)
        obs, _ = e.reset()
        obs = torch.tensor(obs, dtype=torch.float32)
        ep = 0
        for t in range(STEPS_PER_WEEK):
            if force_zero_hints:
                obs[-3:] = 0.0
            with torch.no_grad():
                mean, _ = net.forward(obs.unsqueeze(0))
            obs, r, done, tr, info = e.step(mean.squeeze(0).numpy())
            obs = torch.tensor(obs, dtype=torch.float32)
            ep += info['breaches']
            if done: break
        breaches_list.append(ep)
    return float(np.mean(breaches_list)), float(np.std(breaches_list))

net_hint = ActorCritic()
net_hint.load_state_dict(torch.load('ppo_hint_aware.pth'))
net_hint.eval()

N = 30
with_h, with_std       = eval_hint(net_hint, N, force_zero_hints=False)
without_h, without_std = eval_hint(net_hint, N, force_zero_hints=True)

reduction = (without_h - with_h) / without_h * 100 if without_h > 0 else 0.0

print("="*54)
print("HINT EXPERIMENT — ±5 VM/step (30 weeks, same surges)")
print("="*54)
print(f"  WITH hints active:       {with_h:>8.0f} ± {with_std:.0f} breaches/week")
print(f"  WITHOUT hints (blanked): {without_h:>8.0f} ± {without_std:.0f} breaches/week")
print("="*54)
print(f"\n  Hints reduce breaches by {reduction:.1f}%")

json.dump({'regime': '±5 VM/step', 'with_hints': with_h, 'without_hints': without_h,
           'with_std': with_std, 'without_std': without_std, 'reduction_pct': reduction},
          open('hint_result_dense.json', 'w'), indent=2)
print("Saved hint_result_dense.json")

HINT EXPERIMENT — ±5 VM/step (30 weeks, same surges)
  WITH hints active:          25320 ± 22787 breaches/week
  WITHOUT hints (blanked):    27043 ± 22761 breaches/week

  Hints reduce breaches by 6.4%
Saved hint_result_dense.json
